In [1]:
import ModelCatalog
from utils import model_utils
from dataset.DataLoader import create_dataloaders
from FineTuner import fine_tune
from FineTuneStrategy import FineTuneStrategy

C:\Users\sinat\anaconda3\envs\breast-cancer-agent\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\sinat\anaconda3\envs\breast-cancer-agent\lib\site-packages\torch\utils\cpp_extension.py:28: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import packaging  # type: ignore[attr-defined]


## Default model catalog
For the default catalog I have provided the agent with 5 models. 2 of these are vision transformers and the rest are classical deep convolutional nets.

In [2]:
catalog = ModelCatalog.get_default_catalog()
print(catalog)

Models:
Model name: vit_b_16, accuracy: 81.072, params: 86.6, gflops: 17.56
Model name: swin_s, accuracy: 83.196, params: 49.6, gflops: 8.74
Model name: resnet50, accuracy: 76.13, params: 25.6, gflops: 4.09
Model name: mobilenet_v3_large, accuracy: 74.042, params: 5.5, gflops: 0.22
Model name: densenet161, accuracy: 77.138, params: 28.7, gflops: 7.73



As you can see in the above cell, the param count ranges from 5.5M to 86.6M. The goal was to provide the model with different tools, each capable of achieving differing levels of accuracy with different parameter counts.

In [3]:
models_data = [
    catalog.get_model_data("vit_b_16"),
    catalog.get_model_data("swin_s"),
    catalog.get_model_data("resnet50"),
    catalog.get_model_data("densenet161"),
    catalog.get_model_data("mobilenet_v3_large")
]
models = [model_utils.get_model(model) for model in models_data]
print(models[0])

BreastCancerModel(
  (base_model): VisionTransformer(
    (conv_proj): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (encoder): Encoder(
      (dropout): Dropout(p=0.0, inplace=False)
      (layers): Sequential(
        (encoder_layer_0): EncoderBlock(
          (ln_1): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (self_attention): MultiheadAttention(
            (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
          )
          (dropout): Dropout(p=0.0, inplace=False)
          (ln_2): LayerNorm((768,), eps=1e-06, elementwise_affine=True)
          (mlp): MLPBlock(
            (0): Linear(in_features=768, out_features=3072, bias=True)
            (1): GELU(approximate='none')
            (2): Dropout(p=0.0, inplace=False)
            (3): Linear(in_features=3072, out_features=768, bias=True)
            (4): Dropout(p=0.0, inplace=False)
          )
        )
        (encoder_layer_1): EncoderBlock(
          

In [4]:
train_loader, val_loader, test_loader = create_dataloaders(batch_size=8)

Created Dataframes (800, 2), (100, 2), (100, 2)


In [5]:
vit_lora = fine_tune(models[0],
                FineTuneStrategy.LORA,
                train_loader=train_loader,
                val_loader=val_loader,
                num_epochs=15,
                )
vit_lora

trainable params: 1,769,472 || all params: 87,571,973 || trainable%: 2.0206
class_weights: tensor([0.1548, 0.4595, 1.3293, 1.2407, 1.8157], device='cuda:0')

Starting fine-tuning with strategy: LORA of model vit_b_16.
Device: cuda, Epochs: 15, LR: 0.0001, WD: 0.0001
--------------------------------------------------------------------------------
Epoch 1/15
  Train Loss: 1.6235
  Val Loss: 1.5710
  Val Accuracy: 0.6000
  Val F1 (Weighted): 0.5020
  Val F1 (Macro): 0.1569
  Val Precision: 0.1348
  Val Recall: 0.1875
--------------------------------------------------------------------------------
Epoch 2/15
  Train Loss: 1.5467
  Val Loss: 1.5652
  Val Accuracy: 0.5800
  Val F1 (Weighted): 0.5185
  Val F1 (Macro): 0.2687
  Val Precision: 0.2461
  Val Recall: 0.3314
--------------------------------------------------------------------------------
Epoch 3/15
  Train Loss: 1.4655
  Val Loss: 1.6475
  Val Accuracy: 0.3600
  Val F1 (Weighted): 0.4100
  Val F1 (Macro): 0.1987
  Val Precision: 0.

FineTuneResult(model_name='vit_b_16', fine_tune_strategy='LORA', model_dir="../model_dir/vit_b_16-LORA--LR-'0.0001'--WD-'0.0001'--epochs-'15'", learning_rate=0.0001, weight_decay=0.0001, loss=1.5652234554290771, accuracy=0.58, precision=0.24607142857142855, recall=0.3313988095238095, f1_micro=0.58, f1_macro=0.2686507936507936, f1_weighted=0.5184920634920634, confusion_matrix=[[55, 3, 3, 1, 2], [15, 1, 0, 3, 2], [6, 3, 0, 0, 0], [3, 0, 0, 1, 0], [1, 0, 0, 0, 1]])

In [6]:
resnet_full_ft = fine_tune(models[2],
                          strategy=FineTuneStrategy.FULL_FT,
                          train_loader=train_loader,
                          val_loader=val_loader)
resnet_full_ft

class_weights: tensor([0.1548, 0.4595, 1.3293, 1.2407, 1.8157], device='cuda:0')

Starting fine-tuning with strategy: FULL_FT of model resnet50.
Device: cuda, Epochs: 10, LR: 0.0001, WD: 0.0001
--------------------------------------------------------------------------------
Epoch 1/10
  Train Loss: 1.5778
  Val Loss: 2.4987
  Val Accuracy: 0.0800
  Val F1 (Weighted): 0.0705
  Val F1 (Macro): 0.0802
  Val Precision: 0.1245
  Val Recall: 0.2538
--------------------------------------------------------------------------------
Epoch 2/10
  Train Loss: 1.3227
  Val Loss: 1.7131
  Val Accuracy: 0.4800
  Val F1 (Weighted): 0.4710
  Val F1 (Macro): 0.1889
  Val Precision: 0.1809
  Val Recall: 0.2076
--------------------------------------------------------------------------------
Epoch 3/10
  Train Loss: 0.9761
  Val Loss: 2.2902
  Val Accuracy: 0.4800
  Val F1 (Weighted): 0.4795
  Val F1 (Macro): 0.2701
  Val Precision: 0.2462
  Val Recall: 0.3321
-----------------------------------------------

FineTuneResult(model_name='resnet50', fine_tune_strategy='FULL_FT', model_dir="../model_dir/resnet50-FULL_FT--LR-'0.0001'--WD-'0.0001'--epochs-'10'", learning_rate=0.0001, weight_decay=0.0001, loss=1.713124041373913, accuracy=0.48, precision=0.18085203847915712, recall=0.20758928571428573, f1_micro=0.48, f1_macro=0.18889823380992432, f1_weighted=0.4710260723296888, confusion_matrix=[[39, 23, 2, 0, 0], [12, 9, 0, 0, 0], [5, 2, 0, 0, 2], [2, 2, 0, 0, 0], [1, 1, 0, 0, 0]])

In [7]:
mobilenet_probe = fine_tune(models[4],
                            strategy=FineTuneStrategy.LINEAR_PROBE,
                            train_loader=train_loader,
                            val_loader=val_loader)
mobilenet_probe

class_weights: tensor([0.1548, 0.4595, 1.3293, 1.2407, 1.8157], device='cuda:0')

Starting fine-tuning with strategy: LINEAR_PROBE of model mobilenet_v3_large.
Device: cuda, Epochs: 10, LR: 0.0001, WD: 0.0001
--------------------------------------------------------------------------------
Epoch 1/10
  Train Loss: 1.6017
  Val Loss: 1.6700
  Val Accuracy: 0.0600
  Val F1 (Weighted): 0.0590
  Val F1 (Macro): 0.0491
  Val Precision: 0.1393
  Val Recall: 0.1412
--------------------------------------------------------------------------------
Epoch 2/10
  Train Loss: 1.5598
  Val Loss: 1.7054
  Val Accuracy: 0.0300
  Val F1 (Weighted): 0.0336
  Val F1 (Macro): 0.0238
  Val Precision: 0.0912
  Val Recall: 0.1126
--------------------------------------------------------------------------------
Epoch 3/10
  Train Loss: 1.5220
  Val Loss: 1.7042
  Val Accuracy: 0.0400
  Val F1 (Weighted): 0.0338
  Val F1 (Macro): 0.0280
  Val Precision: 0.1246
  Val Recall: 0.2126
--------------------------------

FineTuneResult(model_name='mobilenet_v3_large', fine_tune_strategy='LINEAR_PROBE', model_dir="../model_dir/mobilenet_v3_large-LINEAR_PROBE--LR-'0.0001'--WD-'0.0001'--epochs-'10'", learning_rate=0.0001, weight_decay=0.0001, loss=1.6058499996478741, accuracy=0.4, precision=0.24916666666666668, recall=0.22306547619047618, f1_micro=0.4, f1_macro=0.23060224089635853, f1_weighted=0.4315966386554622, confusion_matrix=[[31, 18, 3, 2, 10], [8, 8, 0, 0, 5], [6, 3, 0, 0, 0], [1, 1, 0, 1, 1], [2, 0, 0, 0, 0]])